# qwen_ft GenImage Export on Kaggle

This notebook is a **thin wrapper** for step 1-3 of the qwen_ft fine-tuning
flow (`docs/runbook-qwen-finetune-vertex.md`): it verifies the GenImage data
root, runs the balanced subset export script, and uploads the selected images,
manifests, and the label-only training JSONL to the project GCS bucket.

It does **not** implement subset selection, checksum computation, GCS layout,
or training. Those live in the package and the Vertex job.

- Bucket: `gs://aiforensics-qwen-ft-579187260419`
- Protocol: `protocol-a-small`, 100 train and 50 eval images per label per
  generator.
- Enable **Internet** in the Kaggle notebook settings; attach the GenImage
  dataset (layout `<generator>/<split>/{ai,nature}/*`).

## 1. Install the repository and dependencies

The notebook clones this repository into writable storage and installs the
package. Nothing is pinned here; dependency names come from `pyproject.toml`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/ai-image-forensics")
REPO_GIT_URL = "https://github.com/Nnguyen-dev2805/ai-image-forensics.git"

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_GIT_URL, str(REPO_ROOT)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", "."],
    cwd=str(REPO_ROOT),
    check=True,
)
print("repository:", REPO_ROOT)

## 2. Authenticate GCP (Kaggle)

Reads the service-account JSON from the Kaggle Secret
`GOOGLE_APPLICATION_CREDENTIALS`, writes it to a private key file, and
activates it for both `gcloud storage cp` and the Google Cloud Storage client.
**Credential material is never printed.**

In [ ]:
import json
import os
import shlex
import subprocess
import tempfile
from pathlib import Path

from kaggle_secrets import UserSecretsClient

service_account_json = UserSecretsClient().get_secret("GOOGLE_APPLICATION_CREDENTIALS")
service_account_info = json.loads(service_account_json)  # validates JSON before use

KEY_FILE = Path(tempfile.mkstemp(prefix="sa-", suffix=".json")[1])
KEY_FILE.write_text(service_account_json, encoding="utf-8")
os.chmod(KEY_FILE, 0o600)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(KEY_FILE)

activate_cmd = f"gcloud auth activate-service-account --key-file={KEY_FILE}"
subprocess.run(shlex.split(activate_cmd), check=True)
print("gcloud authenticated as:", service_account_info.get("client_email", "<unknown>"))

## 3. Verify the GenImage data root

The export expects the Kaggle dataset layout
`<DATA_ROOT>/<generator>/<train|val>/{ai,nature}/*`. The cell lists the
generator directories it can see; the export script fails loudly when a
configured generator is missing or has too few images.

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/yangsangtai/tiny-genimage")

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"DATA_ROOT does not exist: {DATA_ROOT}. Attach the GenImage dataset "
        "and point DATA_ROOT at its directory."
    )

split_dirs = ("train", "val")
found = [
    entry.name
    for entry in sorted(DATA_ROOT.iterdir())
    if entry.is_dir() and any((entry / split).is_dir() for split in split_dirs)
]
print("generator directories:", len(found))
for name in found:
    print("  -", name)
if not found:
    raise FileNotFoundError(
        f"No <generator>/<split> layout under {DATA_ROOT}; check the attachment."
    )

## 4. Run the export script

Dry-run first (manifests only), then upload with `--upload`. The script writes
`train.csv`, `eval_small.csv`, and `qwen_train.jsonl` into `WORK_DIR`, copies
the selected images to `gs://aiforensics-qwen-ft-579187260419/data/protocol-a-small/`, and uploads the
manifests plus training JSONL alongside them.

In [ ]:
import shlex
import subprocess
import sys

EXPORT_CMD = (
    f"{sys.executable} scripts/export_genimage_qwen_ft_subset.py "
    f"--data-root {DATA_ROOT} "
    f"--bucket-uri gs://aiforensics-qwen-ft-579187260419 "
    "--protocol protocol-a-small "
    "--work-dir /kaggle/working/qwen-ft-export "
    "--seed 70 "
    "--train-per-label 100 "
    "--eval-per-label 50"
)

print("[dry-run]", EXPORT_CMD)
subprocess.run(shlex.split(EXPORT_CMD), cwd=str(REPO_ROOT), check=True)

## 5. Upload to GCS

Re-runs are idempotent per file (`gcloud storage cp` overwrites), but the
selection is deterministic only for the same seed and data root. Do not change
the seed between a partial and a full upload.

In [ ]:
import shlex
import subprocess

subprocess.run(shlex.split(EXPORT_CMD + " --upload"), cwd=str(REPO_ROOT), check=True)

gcs_base = "gs://aiforensics-qwen-ft-579187260419/data/protocol-a-small"
print("uploaded GCS paths:")
for name in ("train.csv", "eval_small.csv"):
    print(f"  {gcs_base}/manifests/{name}")
print(f"  {gcs_base}/qwen_train.jsonl")